In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kvmokshithrao/contract-clauses-dataset/clauses.csv


In [2]:
!pip install -q transformers datasets accelerate peft wandb scikit-learn

In [1]:
import pandas as pd

# Load dataset (upload your CSV to Kaggle first)
df = pd.read_csv("/kaggle/input/datasets/kvmokshithrao/contract-clauses-dataset/clauses.csv")

# Basic checks
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns)

# Show samples
df.head()

Dataset shape: (9915, 2)

Columns: Index(['text', 'label'], dtype='object')


,text,label
0,The term of this Agreement shall be ten (10) y...,effective date
1,Unless earlier terminated otherwise provided t...,effective date
2,The term of this Agreement shall be ten (10) y...,expiration date
3,If Distributor complies with all of the terms ...,renewal term
4,This Agreement is to be construed according to...,governing law


In [2]:
# Sanity Check
print("\nUnique labels:", df["label"].nunique())
print("\nLabel distribution:\n", df["label"].value_counts().head())


Unique labels: 41

Label distribution:
 label
license grant       751
cap on liability    665
audit rights        636
anti-assignment     623
insurance           546
Name: count, dtype: int64


## Label Encoding

In [3]:
from sklearn.preprocessing import LabelEncoder
import json

# Initialize encoder
label_encoder = LabelEncoder()

# Fit and transform labels
df["label_encoded"] = label_encoder.fit_transform(df["label"])

# Create mapping
label_mapping = {
    label: int(idx) 
    for label, idx in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
}

print("Label Mapping:\n")
for k, v in label_mapping.items():
    print(f"{k} → {v}")

#  SAVE mapping
with open("label_mapping.json", "w") as f:
    json.dump(label_mapping, f)

# Check dataset
df.head()

Label Mapping:

affiliate license-licensee → 0
affiliate license-licensor → 1
agreement date → 2
anti-assignment → 3
audit rights → 4
cap on liability → 5
change of control → 6
competitive restriction exception → 7
covenant not to sue → 8
document name → 9
effective date → 10
exclusivity → 11
expiration date → 12
governing law → 13
insurance → 14
ip ownership assignment → 15
irrevocable or perpetual license → 16
joint ip ownership → 17
license grant → 18
liquidated damages → 19
minimum commitment → 20
most favored nation → 21
no-solicit of customers → 22
no-solicit of employees → 23
non-compete → 24
non-disparagement → 25
non-transferable license → 26
notice period to terminate renewal → 27
parties → 28
post-termination services → 29
price restrictions → 30
renewal term → 31
revenue/profit sharing → 32
rofr/rofo/rofn → 33
source code escrow → 34
termination for convenience → 35
third party beneficiary → 36
uncapped liability → 37
unlimited/all-you-can-eat-license → 38
volume restrictio

,text,label,label_encoded
0,The term of this Agreement shall be ten (10) y...,effective date,10
1,Unless earlier terminated otherwise provided t...,effective date,10
2,The term of this Agreement shall be ten (10) y...,expiration date,12
3,If Distributor complies with all of the terms ...,renewal term,31
4,This Agreement is to be construed according to...,governing law,13


## Train-Test Split and HuggingFace Dataset

In [4]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Split dataset
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_encoded"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(train_dataset)
print(test_dataset)

Train size: 7932
Test size: 1983
Dataset({
    features: ['text', 'label', 'label_encoded'],
    num_rows: 7932
})
Dataset({
    features: ['text', 'label', 'label_encoded'],
    num_rows: 1983
})


## Tokenization

In [5]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

# Tokenization function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns
train_dataset = train_dataset.remove_columns(["text", "label"])
test_dataset = test_dataset.remove_columns(["text", "label"])

# Rename label column
train_dataset = train_dataset.rename_column("label_encoded", "labels")
test_dataset = test_dataset.rename_column("label_encoded", "labels")

# Set format for PyTorch
train_dataset.set_format("torch")
test_dataset.set_format("torch")

print(train_dataset)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/7932 [00:00<?, ? examples/s]

Map:   0%|          | 0/1983 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7932
})


## Baseline Model

In [6]:
import torch
from transformers import AutoModelForSequenceClassification

# Load model (pretrained)
model = AutoModelForSequenceClassification.from_pretrained(
    "nlpaueb/legal-bert-base-uncased",
    num_labels=len(label_mapping)
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded on:", device)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

Model loaded on: cuda


## Baseline Evaluation

In [7]:
from sklearn.metrics import accuracy_score, f1_score
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report

model.eval()

predictions = []
true_labels = []

# Loop through test dataset
for batch in tqdm(test_dataset):
    input_ids = batch["input_ids"].unsqueeze(0).to(device)
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    
    logits = outputs.logits
    pred = torch.argmax(logits, dim=1)

    predictions.append(pred.item())
    true_labels.append(labels.item())

# Metrics
accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average="weighted")

print("Baseline Accuracy:", accuracy)
print("Baseline F1 Score:", f1)
print(classification_report(true_labels, predictions))


100%|██████████| 1983/1983 [00:32<00:00, 60.29it/s]

Baseline Accuracy: 0.018658598083711547
Baseline F1 Score: 0.00533638628704858
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        22
           1       0.00      0.00      0.00        14
           2       0.00      0.00      0.00         1
           3       0.00      0.00      0.00       125
           4       0.00      0.00      0.00       127
           5       0.00      0.00      0.00       133
           6       0.01      0.02      0.01        48
           7       0.00      0.00      0.00        25
           8       0.00      0.00      0.00        33
           9       0.00      0.00      0.00         4
          10       0.00      0.00      0.00        27
          11       0.05      0.28      0.08        81
          12       0.00      0.00      0.00        91
          13       0.00      0.00      0.00        89
          14       0.00      0.00      0.00       109
          15       0.00      0.00      0.00        62
  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Setup LoRA and Training Configuration

In [ ]:
import wandb

wandb.login()

wandb.init(project="contract-intelligence", name="legalbert-lora")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments

# LoRA config (same as before, correct)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate=2e-4,          
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,          

    weight_decay=0.01,
    warmup_ratio=0.1,         

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,

    load_best_model_at_end=True,

    report_to="wandb",
)

## Training

In [ ]:
from transformers import Trainer
from sklearn.metrics import accuracy_score, f1_score

# Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    
    return {
        "accuracy": acc,
        "f1": f1,
    }

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()

In [ ]:
# Save LoRA adapters
trainer.model.save_pretrained("/kaggle/working/legalbert-lora")

# Save tokenizer
tokenizer.save_pretrained("/kaggle/working/legalbert-lora")

print("Model saved successfully!")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/legalbert-lora",  # output zip path
    'zip',
    "/kaggle/working/legalbert-lora"   # folder to zip
)

print("Zipped successfully!")